# Assembly
**Megahit**
https://www.metagenomics.wiki/tools/assembly/megahit
- de novo assembly (w/o reference genome)
- aligns/assembles short reads together to reconstruct one 'metagenome'
- assembled contigs are stored in fasta file

In [22]:
# Using trimmed, qc seqs from /trimmed
# separate into groups based on metadata 
    # spp x health status x sampledata - created in reads_counts. groups found in reads_meta
# 1)remove host from sample reads
# 2)remove symbiont and human reads
# 3)concatenate all f and r seqs into single file (1 for f, 1 for r)
# 4)ASSEMBLE reads into contigs (contiguous sequence - joins them together based on read overlap, 
    #and ensures there are no gaps - larger portions of genomes if not all are now together in one sequence)

In [23]:
# based on reads_meta groups, make folders and separate samples out

## Metadata and File Setup

In [36]:
import pandas as pd
import numpy as np
import os 
from pathlib import Path

In [25]:
os.chdir("/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw")

In [31]:
reads_meta = pd.read_csv("reads_meta.csv")
reads_meta.head()

,sampleid,raw,trimmed,pct_yield,Month_year,CollectionDate,Transect,TransectNum,NewTagNum,Species,SampleNum,Health_status,colony_id,group
0,052022_BEL_CBC_T1_10_PSTR,68572842,68449647,99.82,52022.0,5/21/22,CBC30N,1.0,4,PSTR,10.0,Diseased_Margin,T1_4_PSTR,52022_PSTR_Diseased_Margin
1,052022_BEL_CBC_T1_10_PSTR,68572842,68449647,99.82,52022.0,5/21/22,CBC30N,1.0,12,PSTR,10.0,Healthy,T1_12_PSTR,52022_PSTR_Healthy
2,052022_BEL_CBC_T1_11_PSTR,48741905,48660782,99.83,52022.0,5/21/22,CBC30N,1.0,4,PSTR,11.0,Diseased_Tissue,T1_4_PSTR,52022_PSTR_Diseased_Tissue
3,052022_BEL_CBC_T1_11_PSTR,48741905,48660782,99.83,52022.0,5/21/22,CBC30N,1.0,12,PSTR,11.0,Healthy,T1_12_PSTR,52022_PSTR_Healthy
4,052022_BEL_CBC_T1_12_MCAV,166267321,165624856,99.61,52022.0,5/21/22,CBC30N,1.0,8,MCAV,12.0,Diseased_Margin,T1_8_MCAV,52022_MCAV_Diseased_Margin


In [35]:
spp_list=reads_meta['Species'].unique()
print(spp_list)

['PSTR' 'MCAV' 'PAST' 'ORBI' 'MMEA' 'NEG']


In [33]:
reads_meta['group'].unique()

array(['52022_PSTR_Diseased_Margin', '52022_PSTR_Healthy',
       '52022_PSTR_Diseased_Tissue', '52022_MCAV_Diseased_Margin',
       '52022_MCAV_Diseased_Tissue', '52022_PAST_Healthy',
       '52022_OANN_Healthy', '52022_PAST_Diseased_Tissue',
       '52022_MCAV_Healthy', '52022_OFAV_Healthy',
       '52022_PAST_Diseased_Margin', '62019_MMEA_Healthy',
       '62019_PAST_Healthy', '62019_MCAV_Healthy', '102019_PSTR_Healthy',
       '122022_OANN_Diseased_Margin', '122022_PSTR_Healthy',
       '122022_OANN_Healthy', '122022_PSTR_Diseased_Tissue',
       '122022_PSTR_Diseased_Margin', '122022_PAST_Diseased_Margin',
       '122022_OANN_Diseased_Tissue', '122022_PAST_Diseased_Tissue',
       '122022_MCAV_Diseased_Tissue', '122022_PAST_Healthy',
       '122022_MCAV_Healthy', '122022_OFAV_Diseased_Margin',
       '122022_OFAV_Diseased_Tissue', '122022_OFAV_Healthy',
       '122022_MCAV_Diseased_Margin', 'Negative'], dtype=object)

In [ ]:
# make sample list for each spp and group? 

In [ ]:
# common variables to use in scripts
BASE_DIR = "/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw"
STOREREADS = "reads_filtered.txt" # read_count,step,sampleid

In [ ]:
# separating into diff steps 

## SBATCH SCRIPTS

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH --qos=long
#SBATCH -t 168:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/slurm-assembly-%j.out  # %j = job ID

module load conda/latest
conda activate anvio-8
cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw

# set paths for existing bowtie genome indices
MCAV_index=Mcav_DB
MCAV_path="/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/Mcav_genome/"
MMEA_index=Mmea_DB
MMEA_path="/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/Mmea_genome/"
ORBI_index=Ofav_DB
ORBI_path="/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/Ofav_genome/"
# use ssid for past (no PAST host genome - closest relative for genomes we have)
PAST_index=Ssid_DB
PAST_path="/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/Ssid_genome/"
# use cnat for pstr (no PSTR host genome- closest relative for genomes we have)
PSTR_index=Cnat_DB
PSTR_path="/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/Cnat_genome/"
 
# get unique species from list of samples, spp, and groups
spp_list=$(cut -f 2 filtered_sample_groups.txt | tail -n +2 | sort -u)
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/trimmed"

# loop through spp list...
for spp in $spp_list; do
    # make spp folder if it doesn't already exist 
    mkdir -p "$spp"

    # identify samples for each spp to host remove together 
    samples=$(awk -F'\t' -v s="$spp" '$2 == s {print $1}' filtered_sample_groups.txt)

    # copy samples to spp folders
    for id in $samples; do
        if [[ -f "${READSPATH}/${id}_R1_001_val_1.fq" ]] && [[ -f "${READSPATH}/${id}_R2_001_val_2.fq" ]]; then
            # cp "$READSPATH/${id}_R1_001_val_1.fq" "$spp/"
            # cp "$READSPATH/${id}_R2_001_val_2.fq" "$spp/"
            echo "all ${id} files present in $spp"
        else
            echo "Missing files for $spp: $id in $READSPATH"
        fi
    done

    # create file with samplelist and groups for each spp 
    (awk -F'\t' -v s="$spp" '$2 == s' filtered_sample_groups.txt) | cut -f1,3- > $spp/spp_samples 

# 1)remove host from sample reads
# Host seq removal - Thij's script https://github.com/ThijsSt/SCTLD-metagenomes/blob/main/Quality_control_metagenomes.ipynb
    # by specie
    FINALREADS="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed"
    WORKINGPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed/temp"
    mkdir -p $FINALREADS
    mkdir -p $WORKINGPATH

    # assigning path and index variable for each spp       
    spp_index="${spp}_index"
    spp_path="${spp}_path"        
    input_index="${!spp_index}"
    input_path="${!spp_path}"
    
    #skip bowtie index build - already done
    #loop through samples in each spp group
    for id in $samples; do
        #re-align reads back to the index (host genome)
        bowtie2 -p 8 -x $input_path/$input_index -1 $READSPATH/"${id}_R1_001_val_1.fq" -2 $READSPATH/"${id}_R2_001_val_2.fq" -S $WORKINGPATH/"${id}"_mapped_and_unmapped.sam
        
        #convert sam file from bowtie to a bam file for processing
        samtools view -bS $WORKINGPATH/"${id}"_mapped_and_unmapped.sam > $WORKINGPATH/"${id}"_mapped_and_unmapped.bam
        
        #extract only the reads of which both do not match against the host genome
        samtools view -b -f 12 -F 256 $WORKINGPATH/"${id}"_mapped_and_unmapped.bam > $WORKINGPATH/"${id}"_bothReadsUnmapped.bam
        
        # sorts the file so both mates are together and then extracts them back as .fastq files
        samtools sort -n -m 5G -@ 2 $WORKINGPATH/"${id}"_bothReadsUnmapped.bam -o $WORKINGPATH/"${id}"_bothReadsUnmapped_sorted.bam
        samtools fastq -@ 8 $WORKINGPATH/"${id}"_bothReadsUnmapped_sorted.bam \
            -1 $FINALREADS/"${id}"_host_removed_R1.fastq \
            -2 $FINALREADS/"${id}"_host_removed_R2.fastq \
            -0 /dev/null -s /dev/null -n
         if [ $? -eq 0 ]; then
            echo "host removal completed successfully for sample: ${id}"
        else
            echo "host removal encountered an error for sample: ${id}"
            exit 1  
        fi
    done     
done
conda deactivate
echo "Host removal: All samples processed successfully."

# JOB-ID: 53514255
# bash script file name: host_removal

In [ ]:
# run multiple scripts for multiple spp at the same time to make faster 
# start with orbi for second script

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 48:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/slurm-assemblyORBI-%j.out  # %j = job ID

module load conda/latest
conda activate anvio-8
cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw

# set paths for spp & existing bowtie genome indices
spp="ORBI" 
ORBI_index=Ofav_DB
ORBI_path="/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/Ofav_genome/"

# identify samples for each spp to host remove together 
samples=$(awk -F'\t' -v s="$spp" '$2 == s {print $1}' filtered_sample_groups.txt)

# already checked that all samples within the spp are present before proceeding - see old versions on git
# file with samplelist and groups for each spp has already been created

# 1)remove host from sample reads
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/trimmed"
FINALREADS="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed"
WORKINGPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed/temp"
mkdir -p $FINALREADS
mkdir -p $WORKINGPATH

# assigning path and index variable for each spp       
spp_index="${spp}_index"
spp_path="${spp}_path"        
input_index="${!spp_index}"
input_path="${!spp_path}"
    
#loop through samples in each spp (#skip bowtie index build - already done)
for id in $samples; do
    # check if sample has already been completed
    if [[ -f "${FINALREADS}/${id}_host_removed_R1.fastq" ]] && [[ -f "${FINALREADS}/${id}_host_removed_R2.fastq" ]]; then
        echo "${id} already completed"
    else
        # proceed with bowtie if it hasnt been completed
        bowtie2 -p 16 -x $input_path/$input_index \
            -1 "$READSPATH/${id}_R1_001_val_1.fq" \
            -2 "$READSPATH/${id}_R2_001_val_2.fq" | \
            samtools view -@ 6 -b -f 12 -F 256 - > "$WORKINGPATH/${id}_bothReadsUnmapped.bam"
        
        # sorts the file so both mates are together and then extracts them back as .fastq files
        samtools sort -n -m 4G -@ 12 "$WORKINGPATH/${id}_bothReadsUnmapped.bam" -o "$WORKINGPATH/${id}_bothReadsUnmapped_sorted.bam"
        samtools fastq -@ 16 "$WORKINGPATH/${id}_bothReadsUnmapped_sorted.bam" \
            -1 >(bgzip -c > "$FINALREADS/${id}_host_removed_R1.fastq.gz") \
            -2 >(bgzip -c > "$FINALREADS/${id}_host_removed_R2.fastq.gz") \
            -0 /dev/null -s /dev/null -n
         if [ $? -eq 0 ]; then
            echo "host removal completed successfully for sample: ${id}"
            # delete intermediate bam files
            rm -f "$WORKINGPATH/${id}"_*.bam
        else
            echo "host removal encountered an error for sample: ${id}"
            exit 1  
        fi
    fi
done  

conda deactivate
echo "Host removal: All samples processed successfully."

# JOB-ID: 53707659,53723284
# bash script file name: host_removal_orbi

In [ ]:
# repeat PAST

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 48:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/slurm-assemblyPAST-%j.out  # %j = job ID

module load conda/latest
conda activate anvio-8
cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw

# stnd variables
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/trimmed"

# set paths for spp & existing bowtie genome indices
spp="PAST" 
# use ssid for past (no PAST host genome - closest relative for genomes we have)
PAST_index=Ssid_DB
PAST_path="/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/Ssid_genome/"

# identify samples for each spp to host remove together 
samples=$(awk -F'\t' -v s="$spp" '$2 == s {print $1}' filtered_sample_groups.txt)

# already checked that all samples within the spp are present before proceeding - see old versions on git
# file with samplelist and groups for each spp has already been created

# 1)remove host from sample reads
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/trimmed"
FINALREADS="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed"
WORKINGPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed/temp"
mkdir -p $FINALREADS
mkdir -p $WORKINGPATH

# assigning path and index variable for each spp       
spp_index="${spp}_index"
spp_path="${spp}_path"        
input_index="${!spp_index}"
input_path="${!spp_path}"
    
#loop through samples in each spp (#skip bowtie index build - already done)
for id in $samples; do
    # check if sample has already been completed
    if [[ -f "${FINALREADS}/${id}_host_removed_R1.fastq" ]] && [[ -f "${FINALREADS}/${id}_host_removed_R2.fastq" ]]; then
        echo "${id} already completed"
    else
        # proceed with bowtie if it hasnt been completed
        bowtie2 -p 16 -x $input_path/$input_index \
            -1 "$READSPATH/${id}_R1_001_val_1.fq" \
            -2 "$READSPATH/${id}_R2_001_val_2.fq" | \
            samtools view -@ 6 -b -f 12 -F 256 - > "$WORKINGPATH/${id}_bothReadsUnmapped.bam"
        
        # sorts the file so both mates are together and then extracts them back as .fastq files
        samtools sort -n -m 4G -@ 12 "$WORKINGPATH/${id}_bothReadsUnmapped.bam" -o "$WORKINGPATH/${id}_bothReadsUnmapped_sorted.bam"
        samtools fastq -@ 16 "$WORKINGPATH/${id}_bothReadsUnmapped_sorted.bam" \
            -1 > (bgzip -c > "$FINALREADS/${id}_host_removed_R1.fastq.gz") \
            -2 > (bgzip -c > "$FINALREADS/${id}_host_removed_R2.fastq.gz") \
            -0 /dev/null -s /dev/null -n
         if [ $? -eq 0 ]; then
            echo "host removal completed successfully for sample: ${id}"
            # delete intermediate bam files
            rm -f "$WORKINGPATH/${id}"_*.bam
        else
            echo "host removal encountered an error for sample: ${id}"
            exit 1  
        fi
    fi
done  

conda deactivate
echo "Host removal: All samples processed successfully."

# JOB-ID: 53707344, 
# bash script file name: host_removal_past

In [ ]:
# repeat pstr

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 48:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/slurm-assemblyPAST-%j.out  # %j = job ID

module load conda/latest
conda activate anvio-8
cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw

# stnd variables
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/trimmed"

# set paths for spp & existing bowtie genome indices
spp="PSTR" 
# use cnat for pstr (no PSTR host genome- closest relative for genomes we have)
PSTR_index=Cnat_DB
PSTR_path="/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/Cnat_genome/"

# identify samples for each spp to host remove together 
samples=$(awk -F'\t' -v s="$spp" '$2 == s {print $1}' filtered_sample_groups.txt)
mkdir -p "$spp"

# double check that all samples within the spp are present before proceeding
all_present=true
for id in $samples; do
    if [[ -f "${READSPATH}/${id}_R1_001_val_1.fq" ]] && [[ -f "${READSPATH}/${id}_R2_001_val_2.fq" ]]; then
        # don't print anything for each sample
        :
    else
        # If any file is missing, we print the error and flip the flag
        echo "Missing files for $spp: $id in $READSPATH"
        all_present=false
    fi
done

# echo if all samples are present
if [ "$all_present" = true ]; then
    echo "Success: All sample files for $spp are present."
fi

# create file with samplelist and groups for each spp 
(awk -F'\t' -v s="$spp" '$2 == s' filtered_sample_groups.txt) | cut -f1,3- > $spp/spp_samples 

# 1)remove host from sample reads
FINALREADS="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed"
WORKINGPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed/temp"
mkdir -p $FINALREADS
mkdir -p $WORKINGPATH

# assigning path and index variable for each spp       
spp_index="${spp}_index"
spp_path="${spp}_path"        
input_index="${!spp_index}"
input_path="${!spp_path}"
    
#loop through samples in each spp (#skip bowtie index build - already done)
for id in $samples; do
    #re-align reads back to the index (host genome)
    #extract only the reads of which both do NOT match against the host genome
    bowtie2 -p 16 -x $input_path/$input_index \
        -1 "$READSPATH/${id}_R1_001_val_1.fq" \
        -2 "$READSPATH/${id}_R2_001_val_2.fq" | \
        samtools view -@ 6 -b -f 12 -F 256 - > "$WORKINGPATH/${id}_bothReadsUnmapped.bam"
    
    # sorts the file so both mates are together and then extracts them back as .fastq files
    samtools sort -n -m 4G -@ 12 "$WORKINGPATH/${id}_bothReadsUnmapped.bam" -o "$WORKINGPATH/${id}_bothReadsUnmapped_sorted.bam"
    samtools fastq -@ 16 "$WORKINGPATH/${id}_bothReadsUnmapped_sorted.bam" \
        -1 "$FINALREADS/${id}_host_removed_R1.fastq" \
        -2 "$FINALREADS/${id}_host_removed_R2.fastq" \
        -0 /dev/null -s /dev/null -n
     if [ $? -eq 0 ]; then
        echo "host removal completed successfully for sample: ${id}"
    else
        echo "host removal encountered an error for sample: ${id}"
        exit 1  
    fi
done     
conda deactivate
echo "Host removal: All samples processed successfully."

# JOB-ID: 53707348
# bash script file name: host_removal_pstr

In [ ]:
# will run read count scripts separately 

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH --qos=long
#SBATCH -t 168:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/slurm-assembly2-%j.out  # %j = job ID

# 2)remove symbiont and human seqs using fastq screen 

cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw
module load Bowtie2/2.4.5-GCC-11.3.0
module load conda/latest
conda activate fastq_screen

FASTQSCREEN='/home/brooke_sienkiewicz_student_uml_edu/.conda/envs/fastq_screen/share/fastq-screen-0.15.3-0'

spp_list=$(cut -f 2 filtered_sample_groups.txt | tail -n +2 | sort -u)
for spp in $spp_list; do
    OUTPUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/sym_human_removed"
    READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed"
    mkdir -p "$OUTPUTDIR"
    if [ $? -ne 0 ]; then
        echo "Error: Failed to create output directory $OUTPUTDIR"
        exit 1
    fi
    
    while IFS= read -r SAMPLEID; do
        $FASTQSCREEN/fastq_screen --nohits --aligner bowtie2 --conf $FASTQSCREEN/fastq_screen.conf --outdir $OUTPUTDIR \
        $READSPATH/"${SAMPLEID}"_host_removed_R1.fastq $READSPATH/"${SAMPLEID}"_host_removed_R2.fastq;
         if [ $? -eq 0 ]; then
                echo "fastq_screen completed successfully for sample: $SAMPLEID"
            else
                echo "fastq_screen encountered an error for sample: $SAMPLEID"
                exit 1
            fi
        # --nohits = output reads do not map to any genomes
    done < $spp/spp_samples
done

conda deactivate
echo "Symbiont, host removal: All samples processed successfully."

# 2)concatenate all f and r seqs into single file (1 for f, 1 for r)
    # by group
conda activate assembly

for spp in $spp_list; do
    cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}
    WORKINGPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/sym_human_removed"
    # get list of unique groups
    spp_groups=$(cut -f 2 spp_samples | sort -u)

    #loop each group
    for g in $spp_groups; do
        OUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/${g}"
        mkdir -p $OUTDIR
        # create list of sampleids by specie x group
        (awk -F'\t' -v g="$g" '$2 == g {print $1}' spp_samples) > $OUTDIR/group_samples
        
        while IFS= read -r SAMPLEID; do
            # Construct the file paths for forward and reverse reads
            FORWARD_READ="$WORKINGPATH/${SAMPLEID}_host_removed_R1.fastq.tagged_filter.fastq.gz"
            REVERSE_READ="$WORKINGPATH/${SAMPLEID}_host_removed_R2.fastq.tagged_filter.fastq.gz"
        
            # Check if the files exist before concatenating
            if [ -e "$FORWARD_READ" ]; then
                zcat "$FORWARD_READ" >> "$OUTDIR/${g}_reads_R1_ALL.fastq"
            else
                echo "Forward read file not found for sample $SAMPLEID"
            fi
        
            if [ -e "$REVERSE_READ" ]; then
                zcat "$REVERSE_READ" >> "$OUTDIR/${g}_reads_R2_ALL.fastq"
            else
                echo "Reverse read file not found for sample $SAMPLEID"
            fi
        done < $OUTDIR/group_samples # finish group samples
    done # finish specie x groups
done # finish all spp 

conda deactivate 
cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw

# 3)ASSEMBLE reads into contigs (contiguous sequence - joins them together based on read overlap, and ensures there are no gaps
conda activate assembly
for spp in $spp_list; do
    for g in $spp_groups; do
        OUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/${g}"
    
        megahit --presets meta-large \
        -1 "$OUTDIR"/"$g"_reads_R1_ALL.fastq \
        -2 "$OUTDIR"/"$g"_reads_R2_ALL.fastq \
        --keep-tmp-files \
        -o megahit_host_removed --out-prefix $g \
        --continue
    done
done
#this one has to make the directory; will fail if it already exists

# JOB-ID:
# bash script file name: assembly_pt2